# 2.4 — Recherche dans le texte avec les expressions régulières (`re`)

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- rechercher un motif dans une chaîne de caractères ;
- récupérer la première occurrence ou toutes les occurrences ;
- obtenir les positions des correspondances ;
- utiliser les chiffres, les caractères génériques, les classes et les ancres ;
- capturer des groupes dans un motif ;
- appliquer ces techniques à des textes en anglais et en arabe.

**Idée clé :** une expression régulière (*regular expression* ou *regex*) décrit un motif à rechercher dans un texte.


## 1. Importer le module `re`

`re` est un module standard de Python : il n’a pas besoin d’être installé avec `pip`.

Nous utilisons des chaînes brutes (`r"..."`) pour écrire les motifs. Cela évite que Python interprète certaines barres obliques inverses avant le moteur regex.

In [ ]:
import re

print("Module re prêt à être utilisé.")

## 2. Rechercher la première occurrence avec `re.search()`

`re.search(pattern, text)` recherche la **première occurrence** du motif n’importe où dans le texte.

- Si une correspondance existe, la fonction renvoie un objet `Match`.
- Si aucune correspondance n’existe, elle renvoie `None`.
- `group()` retourne le texte trouvé.
- `span()` retourne les positions de début et de fin, sous la forme `(début, fin)`. La position de fin est exclue.

In [ ]:
pattern = r"phone"
text = "The agent's phone number is 408-555-1234. Call soon!"

match = re.search(pattern, text)

if match:
    print("Texte trouvé :", match.group())
    print("Positions :", match.span())
    print("Début :", match.start())
    print("Fin :", match.end())
else:
    print("Aucune correspondance.")

### Comprendre les indices

Dans Python, les indices commencent à `0`. Pour le mot `phone`, `start()` donne l’indice du premier caractère et `end()` donne l’indice situé juste après le dernier caractère.

**Attention :** appeler `match.group()` sans vérifier `match` peut provoquer une erreur si le motif n’existe pas dans le texte.

In [ ]:
# Exemple sans correspondance : search() renvoie None
match = re.search(r"email", text)

print(match)  # None

## 3. Trouver toutes les occurrences avec `re.findall()`

`re.findall()` renvoie une liste contenant toutes les correspondances, dans l’ordre. Si le motif ne trouve rien, le résultat est une liste vide.

In [ ]:
text = """
A telephone is a telecommunications device. A telephone permits two or more users
to conduct a conversation when they are too far apart to be heard directly.
"""

pattern = r"telephone"

matches = re.findall(pattern, text)

print("Occurrences :", matches)
print("Nombre d'occurrences :", len(matches))

## 4. Obtenir les correspondances et leurs positions avec `re.finditer()`

`re.finditer()` renvoie un itérateur d’objets `Match`. C’est utile lorsque l’on veut connaître à la fois le texte trouvé et sa position.

In [ ]:
for match in re.finditer(r"telephone", text):
    print(f"Texte : {match.group()!r}, positions : {match.span()}")

## 5. Rechercher dans un texte en arabe

Les expressions régulières peuvent aussi rechercher du texte arabe. Le motif doit contenir les caractères que l’on souhaite repérer. Comme pour les autres langues, `search()` ne renvoie que la première occurrence.

In [ ]:
text_ar = """
ورقة أينشتاين العلمية الثالثة كانت عن "نظرية النسبية الخاصة".
اقترحها ونشرها في 26 سبتمبر 1905.
"""

pattern_ar = r"نسبية"
match_ar = re.search(pattern_ar, text_ar)

if match_ar:
    print("Correspondance :", match_ar.group())
    print("Positions :", match_ar.span())
else:
    print("Aucune correspondance.")

## 6. Extraire des numéros de téléphone

### 6.1 Décrire le motif

Le motif `\d` représente un chiffre. Le quantificateur `{3}` signifie « exactement trois fois ».

Ainsi, `\d{3}-\d{3}-\d{4}` décrit un format tel que `408-555-1234`.

Cette expression vérifie uniquement la **forme** du numéro ; elle ne garantit pas que le numéro existe réellement.

In [ ]:
text = "My telephone number is 408-555-1234 and Mohamed's phone is 856-987-6632."

# \d{3} = trois chiffres ; les tirets sont des caractères littéraux
phone_pattern = r"\d{3}-\d{3}-\d{4}"

# Première correspondance
phone_match = re.search(phone_pattern, text)
if phone_match:
    print("Premier numéro :", phone_match.group())

# Toutes les correspondances
all_phones = re.findall(phone_pattern, text)
print("Tous les numéros :", all_phones)

### 6.2 Les groupes de capture

Les parenthèses `(...)` créent des groupes. Ils permettent d’extraire des parties d’une correspondance.

In [ ]:
phone_pattern = r"(\d{3})-(\d{3})-(\d{4})"
result = re.search(phone_pattern, text)

if result:
    print("Numéro complet :", result.group(0))
    print("Groupe 1 :", result.group(1))  # Les trois premiers chiffres
    print("Groupe 2 :", result.group(2))  # Les trois chiffres suivants
    print("Groupe 3 :", result.group(3))  # Les quatre derniers chiffres

**À retenir :** `group(0)` correspond à la correspondance complète. Avec trois groupes de capture, seuls `group(1)`, `group(2)` et `group(3)` sont disponibles. Demander `group(4)` provoquerait une `IndexError`.

## 7. Alternatives avec `|`

Le symbole `|` signifie « ou ». Le motif `man|woman` correspond à `man` ou à `woman`.

La recherche est sensible à la casse par défaut : `Man` et `man` sont différents.

In [ ]:
print(re.search(r"man|woman", "This man was here.").group())
print(re.search(r"man|woman", "This woman was here.").group())

# Avec IGNORECASE, les majuscules et minuscules sont ignorées
match = re.search(r"python|nltk", "I am learning Python.", flags=re.IGNORECASE)
print(match.group() if match else "Aucune correspondance")

## 8. Le caractère générique `.`

Le point `.` correspond généralement à un caractère quelconque, sauf le saut de ligne (par défaut).

- `r".at"` : un caractère suivi de `at`, par exemple `cat`, `hat` ou `sat`.
- `r"...at"` : trois caractères suivis de `at`, donc cinq caractères au total.

In [ ]:
sentence = "The cat in the hat sat here."

print(re.findall(r".at", sentence))
print(re.findall(r"...at", "The bat went splat"))

## 9. Classes de caractères

| Motif | Signification |
|---|---|
| `\d` | Un chiffre |
| `\D` | Un caractère qui n’est pas un chiffre |
| `\w` | Un caractère de mot (lettre, chiffre ou `_`, selon Unicode) |
| `\W` | Un caractère qui n’est pas un caractère de mot |
| `\s` | Un espace blanc : espace, tabulation ou saut de ligne |
| `\S` | Un caractère qui n’est pas un espace blanc |
| `[abc]` | Un caractère parmi `a`, `b` ou `c` |
| `[^abc]` | Un caractère qui n’est pas `a`, `b` ou `c` |

Le signe `^` signifie « négation » lorsqu’il est placé au début d’une classe entre crochets.

In [ ]:
sentence = "The bat went splat *plat"

# \S+ = une suite d'au moins un caractère non blanc, suivie de 'at'
print(re.findall(r"\S+at", sentence))

# Extraire les caractères qui ne sont pas des chiffres
print(re.findall(r"\D", "A1B2C3"))

## 10. Ancres : début et fin du texte

- `^` signifie le début de la chaîne (ou le début d’une ligne avec certaines options).
- `$` signifie la fin de la chaîne (ou la fin d’une ligne avec certaines options).

Ces ancres ne sélectionnent pas un caractère : elles indiquent une position.

In [ ]:
# Le texte commence par un chiffre
print(bool(re.search(r"^\d", "1 is the loneliest number.")))  # True

# Le texte se termine par un chiffre
print(bool(re.search(r"\d$", "This ends with a number 28")))  # True

# Extraire une suite de chiffres située à la fin
print(re.findall(r"\d+$", "Chapter 12"))  # ['12']

## 11. Retirer certains éléments d’un texte

Pour extraire les parties qui ne sont pas des chiffres, `[^\d]+` signifie « une suite d’un ou plusieurs caractères qui ne sont pas des chiffres ».

Pour nettoyer la ponctuation, il faut définir explicitement les caractères à retirer. Une classe de caractères ne supprime pas automatiquement toute la ponctuation existante.

In [ ]:
phrase = "There are 3 numbers 34 inside 5 this sentence."

# Récupérer les segments séparés par les chiffres
print(re.findall(r"[^\d]+", phrase))

test_phrase = "This is a string! But it has punctuation. How can we remove it?"

# Séparer les morceaux en excluant ponctuation et espaces ciblés
parts = re.findall(r"[^!.? ]+", test_phrase)
clean = " ".join(parts)

print(clean)

## 12. Rechercher les mots contenant un trait d’union

Le motif `r"[\w]+-[\w]+"` recherche une séquence de caractères de mot, un trait d’union, puis une autre séquence de caractères de mot.

In [ ]:
text = "Find the hyphen-words here. Some words are long-ish."

hyphen_words = re.findall(r"[\w]+-[\w]+", text)
print(hyphen_words)

## 13. Exemples en arabe

Les classes Unicode de Python fonctionnent aussi sur le texte arabe. `\S+` recherche une suite de caractères non blancs ; le motif `..ات` recherche deux caractères quelconques suivis de `ات`.

**Limite :** ces motifs sont purement formels. Ils ne comprennent pas nécessairement la morphologie arabe et peuvent retourner des résultats qui ne sont pas des mots pertinents.

In [ ]:
text_ar = "رقم هاتفي هو 586-986-9965 و رقم هاتف محمد هو 826-476-9785."

# Extraire les numéros au format attendu
phones_ar = re.findall(r"\d{3}-\d{3}-\d{4}", text_ar)
print("Numéros :", phones_ar)

# Chercher les séquences qui se terminent par « ات »
sentence_ar = "ذهبت الطالبات لحضور عدد من المحاضرات في جامعاتهن"
print(re.findall(r"\S+ات", sentence_ar))

### 13.1 Ponctuation arabe

Il faut inclure explicitement les signes de ponctuation arabes que l’on souhaite séparer, par exemple `؟` pour le point d’interrogation arabe.

In [ ]:
text_ar = "ماذا ؟ كيف تجرؤ !! هل تتخيل عاقبة هذا الأمر ؟"

# Retirer les signes choisis et les espaces en séparant les mots
parts_ar = re.findall(r"[^؟،!. ]+", text_ar)
clean_ar = " ".join(parts_ar)

print(clean_ar)

## 14. Résumé des principales fonctions

| Fonction | Utilisation |
|---|---|
| `re.search(pattern, text)` | Première correspondance |
| `re.findall(pattern, text)` | Liste de toutes les correspondances |
| `re.finditer(pattern, text)` | Itérateur d’objets Match avec positions |
| `match.group()` | Texte trouvé |
| `match.group(1)` | Contenu du premier groupe capturé |
| `match.span()` | Positions de début et de fin |
| `match.start()` | Position de début |
| `match.end()` | Position de fin |

## 15. Exercices

1. Recherchez le premier mot `AI` dans : `AI is useful. NLP is a field of AI.`
2. Trouvez toutes les occurrences de `NLP` dans une phrase qui contient ce terme trois fois.
3. Extrayez tous les nombres de : `The course has 18 teams and 3 projects per team.`
4. Extrayez toutes les adresses e-mail d’un texte en utilisant un motif simple.
5. Dans le texte `Call 123-456-7890 or 987-654-3210`, extrayez les deux numéros.
6. Dans un texte arabe, recherchez toutes les occurrences du mot `الجامعة`.
7. Expliquez la différence entre `search()`, `findall()` et `finditer()`.

### Solutions indicatives pour les exercices 3 et 5


In [ ]:
# Exercice 3 : extraire les nombres
text = "The course has 18 teams and 3 projects per team."
print(re.findall(r"\d+", text))  # ['18', '3']

# Exercice 5 : extraire les numéros
text = "Call 123-456-7890 or 987-654-3210"
print(re.findall(r"\d{3}-\d{3}-\d{4}", text))

## À retenir

- Une regex décrit un motif ; elle ne comprend pas nécessairement le sens du texte.
- Vérifiez toujours le résultat de `re.search()` avant d’appeler `.group()`.
- Utilisez `findall()` pour obtenir les correspondances sous forme de liste.
- Les groupes `(...)` permettent de récupérer des parties d’un motif.
- Testez vos motifs sur plusieurs exemples, notamment les cas où aucune correspondance n’existe.
